In [3]:
import os
import shutil
import random

original_data_path = "D:/VS Code/Coding/Deepfake_Project/Phase1_Preprocessing/processed_data"
test_sample_path = "D:/VS Code/Coding/Deepfake_Project/Phase2_Training/test_sample"

for category in ['real', 'fake']:
    os.makedirs(os.path.join(test_sample_path, category), exist_ok=True)
    
    source_folder = os.path.join(original_data_path, category)
    all_images = os.listdir(source_folder)
    
    sample_images = random.sample(all_images, 200)
    
    print(f"Copying 200 {category} images for testing...")
    for img_name in sample_images:
        shutil.copy(os.path.join(source_folder, img_name), 
                    os.path.join(test_sample_path, category, img_name))

print("Small Dataset test wala dataset", test_sample_path)

Copying 200 real images for testing...
Copying 200 fake images for testing...
Small Dataset test wala dataset D:/VS Code/Coding/Deepfake_Project/Phase2_Training/test_sample


In [4]:
import os
import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

IMG_SIZE = (224, 224)
BATCH_SIZE = 8 
EPOCHS = 3
BASE_PATH = "D:/VS Code/Coding/Deepfake_Project/Phase2_Training/test_sample"
CHECKPOINT_PATH = "D:/VS Code/Coding/Deepfake_Project/Phase2_Training/test_best_model.h5"

train_datagen = ImageDataGenerator(rescale=1./255, rotation_range=20, horizontal_flip=True, validation_split=0.2)

train_generator = train_datagen.flow_from_directory(BASE_PATH, target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='binary', subset='training')
val_generator = train_datagen.flow_from_directory(BASE_PATH, target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='binary', subset='validation')

# If power went out and want to RESUME, uncomment the line below:
# model = tf.keras.models.load_model(CHECKPOINT_PATH) 

def build_model():
    base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
    base_model.trainable = False
    x = GlobalAveragePooling2D()(base_model.output)
    x = Dense(512, activation='relu')(x)
    x = Dropout(0.5)(x)
    predictions = Dense(1, activation='sigmoid')(x)
    return Model(inputs=base_model.input, outputs=predictions)

model = build_model() # Comment this out if you are resuming from a loaded model

model.compile(optimizer=Adam(learning_rate=0.0001), loss='binary_crossentropy', metrics=['accuracy'])

early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True, verbose=1)

checkpoint = ModelCheckpoint(
    filepath=CHECKPOINT_PATH,
    monitor='val_accuracy',
    save_best_only=True,   
    mode='max',
    verbose=1
)

print("Starting Phase 2 with Checkpointing enabled...")
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS,
    callbacks=[early_stop, checkpoint], 
    verbose=1
)

model.save("D:/VS Code/Coding/Deepfake_Project/Phase2_Training/artifact_lens_final.h5")

Found 640 images belonging to 2 classes.
Found 160 images belonging to 2 classes.
Starting Phase 2 with Checkpointing enabled...
Epoch 1/3
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 405ms/step - accuracy: 0.4837 - loss: 0.7771
Epoch 1: val_accuracy improved from None to 0.50000, saving model to D:/VS Code/Coding/Deepfake_Project/Phase2_Training/test_best_model.h5



Epoch 1: finished saving model to D:/VS Code/Coding/Deepfake_Project/Phase2_Training/test_best_model.h5
80/80 ━━━━━━━━━━━━━━━━━━━━ 51s 550ms/step - accuracy: 0.4812 - loss: 0.7818 - val_accuracy: 0.5000 - val_loss: 0.6998
Epoch 2/3
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 391ms/step - accuracy: 0.5017 - loss: 0.7533
Epoch 2: val_accuracy did not improve from 0.50000
80/80 ━━━━━━━━━━━━━━━━━━━━ 39s 490ms/step - accuracy: 0.5141 - loss: 0.7494 - val_accuracy: 0.5000 - val_loss: 0.7103
Epoch 3/3
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 408ms/step - accuracy: 0.4915 - loss: 0.7227
Epoch 3: val_accuracy improved from 0.50000 to 0.59375, saving model to D:/VS Code/Coding/Deepfake_Project/Phase2_Training/test_best_model.h5



Epoch 3: finished saving model to D:/VS Code/Coding/Deepfake_Project/Phase2_Training/test_best_model.h5
80/80 ━━━━━━━━━━━━━━━━━━━━ 41s 513ms/step - accuracy: 0.4875 - loss: 0.7311 - val_accuracy: 0.5938 - val_loss: 0.6870
Restoring model weights from the end of the best epoch: 3.
